# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
from ollama import chat

MODEL = "llama3.2:latest"

messages = [
    {"role": "user", "content": "Explain recursion simply"}
]

response = chat(
    model=MODEL,
    messages=messages
)

print(response["message"]["content"])

Recursion is a programming concept where a function calls itself repeatedly until it reaches a base case that stops the recursion.

Think of recursion like a set of Russian nesting dolls. Each doll has a smaller version of itself inside, and when you open one up, you find another smaller version inside, and so on.

Here's a simple example:

1. You have a function called "factorial" that calculates the product of all numbers from 1 to n (e.g., factorial(5) = 1*2*3*4*5).
2. The function calls itself with a smaller value of n (e.g., factorial(4)) until it reaches a base case (n=0 or n=1), which returns a pre-calculated result.
3. When the function returns to its original call, it combines the results from the recursive calls to calculate the final answer.

Recursion is useful when:

* You need to solve problems that have a recursive structure, like calculating factorials or tree traversals.
* You want to write elegant and concise code without using loops.

However, recursion can also be i

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://iamharshith.vercel.app/"))


Here is the list of links on the website https://iamharshith.vercel.app/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#hero
#about
#skills
#projects
#experience
#achievements
#contact
#contact
#projects
/resume.pdf
https://urban-digital-twins-track3.vercel.app
https://github.com/harshith-murali/urban-digital-twins-track3
https://url-shortner-silk-nine.vercel.app
https://github.com/harshith-murali/url-shortner
https://open-verse-arcjet.vercel.app/
https://github.com/harshith-murali/open-verse-arcjet
https://auth-nextjs-theta-seven.vercel.app
https://github.com/harshith-murali/auth-nextjs
http://credly.com/badges/4f7fd64b-9ca4-4ab6-95c3-d27a8402cb1a
https://github.com/harshith-murali
https://leetcode.com/u/geekycoder11
https://github.com/harshith-murali
https://linkedin.com/in/harshith-m-dev
mailto:mhar

In [7]:
import json
from ollama import chat

MODEL = "llama3.2:latest"

def select_relevant_links(url):
    response = chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": link_system_prompt
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        format="json"
    )

    result = response["message"]["content"]
    links = json.loads(result)

    return links

In [8]:
select_relevant_links("https://iamharshith.vercel.app/")

{'links': [{'type': 'about page',
   'url': 'https://iamharshith.vercel.app/#about'},
  {'type': 'Projects/Portfolio',
   'url': 'https://iamharshith.vercel.app/#projects'},
  {'type': 'Experience/Careers',
   'url': 'https://iamharshith.vercel.app/#experience'},
  {'type': 'Achievements',
   'url': 'https://iamharshith.vercel.app/#achievements'},
  {'type': 'LinkedIn Profile',
   'url': 'https://linkedin.com/in/harshith-m-dev'},
  {'type': 'GitHub Profile', 'url': 'https://github.com/harshith-murali'}]}

In [9]:
import json
from ollama import chat

MODEL = "llama3.2:latest"

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")

    response = chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": link_system_prompt
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        format="json"
    )

    result = response["message"]["content"]
    links = json.loads(result)

    print(f"Found {len(links['links'])} relevant links")

    return links

In [10]:
select_relevant_links("https://iamharshith.vercel.app/")

Selecting relevant links for https://iamharshith.vercel.app/ by calling llama3.2:latest
Found 4 relevant links


{'links': [{'type': 'company page', 'url': 'https://iamharshith.vercel.app/'},
  {'type': 'GitHub profile', 'url': 'https://github.com/harshith-murali'},
  {'type': 'LinkedIn profile',
   'url': 'https://linkedin.com/in/harshith-m-dev'},
  {'type': 'Resume', 'url': 'https://iamharshith.vercel.app/#resume.pdf'}]}

In [11]:
select_relevant_links("https://iamharshith.vercel.app/")

Selecting relevant links for https://iamharshith.vercel.app/ by calling llama3.2:latest
Found 8 relevant links


{'links': [{'type': 'company page', 'url': 'https://iamharshith.vercel.app/'},
  {'type': 'about page', 'url': 'https://iamharshith.vercel.app/#about'},
  {'type': 'achievements page',
   'url': 'https://iamharshith.vercel.app/#achievements'},
  {'type': 'experience page',
   'url': 'https://iamharshith.vercel.app/#experience'},
  {'type': 'projects page', 'url': 'https://iamharshith.vercel.app/#projects'},
  {'type': 'contact page', 'url': 'https://iamharshith.vercel.app/#contact'},
  {'type': 'github profile', 'url': 'https://github.com/harshith-murali'},
  {'type': 'linkedin profile',
   'url': 'https://linkedin.com/in/harshith-m-dev'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://vercel.com"))

Selecting relevant links for https://vercel.com by calling llama3.2:latest
Found 6 relevant links
## Landing Page:


URL: https://vercel.com

TITLE:
Vercel: Build and deploy the best web experiences with the AI Cloud

CONTENT:
Vercel: Build and deploy the best web experiences with the AI Cloud Skip to content Products AI Cloud AI Gateway One endpoint, all your models Sandbox Isolated, safe code execution Vercel Agent An agent that knows your stack AI SDK The AI Toolkit for TypeScript v0 Build applications with AI Core Platform CI/CD Helping teams ship 6× faster Content Delivery Fast, scalable, and reliable Fluid Compute Servers, in serverless form Workflow Long-running workflows at scale Observability Trace every step Security Bot Management Scalable bot protection BotID Invisible CAPTCHA Platform Security DDoS Protection, Firewall Web Application Firewall Granular, custom protection Resources Company Customers Trusted by the best teams Blog The latest posts and changes Changelog See w

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("Tesla", "https://www.tesla.com")

Selecting relevant links for https://www.tesla.com by calling llama3.2:latest
Found 4 relevant links


'\nYou are looking at a company called: Tesla\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nCould not fetch https://www.tesla.com\n## Relevant Links:\n\n\n### Link: about page\nCould not fetch https://www.tesla.com/about/\n\n### Link: company overview\nCould not fetch https://www.tesla.com/company-overview/\n\n### Link: careers/jobs\nCould not fetch https://www.tesla.com/careers/\n\n### Link: contact us\nCould not fetch https://www.tesla.com/contact-us/'

In [17]:
from ollama import chat
from IPython.display import Markdown, display

MODEL = "llama3.2:latest"

def create_brochure(company_name, url):
    response = chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ]
    )

    result = response["message"]["content"]

    display(Markdown(result))

In [19]:
create_brochure("GitHub", "https://github.com")

Selecting relevant links for https://github.com by calling llama3.2:latest
Found 13 relevant links


# About GitHub

At GitHub, we believe that the world needs a better place to build software. We're on a mission to empower people all over the world to do amazing things with code.

## Our Story

GitHub was founded in 2008 by Tom Preston-Werner, Chris Wanstrath, and PJ Hyett with the goal of creating an open source alternative to traditional version control systems like Subversion and CVS. Since then, we've grown into one of the largest software development platforms in the world, serving over 100 million developers who use our platform every day.

## Our Culture

Our culture is built on the principles of openness, collaboration, and community. We believe that the best way to build great software is by working together with others. That's why we've created a platform that allows developers from all over the world to share their code, collaborate on projects, and learn from each other.

We're also committed to making technology more accessible and inclusive for everyone. That's why we offer free and open-source solutions for developers of all levels, as well as resources and support to help them get started.

## Our Customers

Our customers are the developers who use our platform every day. They come from all over the world, working on a wide range of projects, from apps and websites to games and artificial intelligence systems.

Some of our notable customers include:

* Airbnb
* Facebook
* Twitter
* Instagram
* Dropbox
* Tesla

## Our Careers

If you're passionate about building great software and want to join a company that's making a difference in the world, we'd love to hear from you. We offer a wide range of career opportunities across our platform, including:

* Developer roles: We hire developers who are passionate about coding and want to build amazing things with code.
* Engineer roles: We also hire engineers who specialize in areas like AI, machine learning, and security.
* Support roles: If you're great at helping people, we'd love to hear from you. Our support team is responsible for helping our customers get the most out of our platform.

## Join Us

If you're ready to join a company that's changing the world one line of code at a time, check out our careers page and start exploring our open positions today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
from ollama import chat
from IPython.display import Markdown, display, update_display

MODEL = "llama3.2:latest"

def stream_brochure(company_name, url):
    stream = chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ],
        stream=True
    )

    response = ""

    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        content = chunk["message"]["content"]

        response += content

        update_display(
            Markdown(response),
            display_id=display_handle.display_id
        )

In [22]:
stream_brochure("NVIDIA", "https://www.nvidia.com")

Selecting relevant links for https://www.nvidia.com by calling llama3.2:latest


KeyError: 'links'

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>